# Agent Communication Patterns

## Overview

In multi-agent systems, communication is crucial for coordination. This notebook explores different communication patterns and their trade-offs.

### Topics Covered:

1. Point-to-Point Communication
2. Broadcast Communication
3. Publish-Subscribe Pattern
4. Request-Reply Pattern
5. Blackboard Pattern
6. Contract Net Protocol

In [ ]:
# Setup
import sys
sys.path.append('..')

from typing import Dict, List, Any, Optional, Callable
from dataclasses import dataclass, field
from datetime import datetime
from enum import Enum
import json

from utils.visualization import (
    visualize_communication_flow,
    visualize_agent_graph,
    plot_agent_interactions
)

print("✓ Setup complete!")

## 1. Point-to-Point Communication

The simplest pattern: Agent A sends a message directly to Agent B.

In [ ]:
@dataclass
class Message:
    """Enhanced message with metadata."""
    sender: str
    recipient: str
    content: str
    message_type: str = "info"
    timestamp: datetime = field(default_factory=datetime.now)
    metadata: Dict[str, Any] = field(default_factory=dict)
    
    def to_dict(self):
        return {
            'from': self.sender,
            'to': self.recipient,
            'content': self.content,
            'type': self.message_type
        }


class CommunicatingAgent:
    """Agent with communication capabilities."""
    
    def __init__(self, name: str):
        self.name = name
        self.inbox: List[Message] = []
        self.outbox: List[Message] = []
        
    def send(self, recipient: str, content: str, message_type: str = "info") -> Message:
        """Send a message to another agent."""
        msg = Message(
            sender=self.name,
            recipient=recipient,
            content=content,
            message_type=message_type
        )
        self.outbox.append(msg)
        print(f"📤 [{self.name}] → [{recipient}]: {content[:60]}...")
        return msg
        
    def receive(self, message: Message):
        """Receive a message from another agent."""
        self.inbox.append(message)
        print(f"📥 [{self.name}] ← [{message.sender}]: {message.content[:60]}...")
        
    def get_messages(self, message_type: Optional[str] = None) -> List[Message]:
        """Retrieve messages, optionally filtered by type."""
        if message_type:
            return [msg for msg in self.inbox if msg.message_type == message_type]
        return self.inbox


# Example: Point-to-point communication
agent_a = CommunicatingAgent("ResearchAgent")
agent_b = CommunicatingAgent("AnalysisAgent")

# A sends to B
msg = agent_a.send("AnalysisAgent", "Here are 100 research papers on neural architectures")
agent_b.receive(msg)

# B responds to A
response = agent_b.send("ResearchAgent", "Analysis complete. Found 3 key trends.")
agent_a.receive(response)

print(f"\n📊 Stats:")
print(f"  ResearchAgent - Sent: {len(agent_a.outbox)}, Received: {len(agent_a.inbox)}")
print(f"  AnalysisAgent - Sent: {len(agent_b.outbox)}, Received: {len(agent_b.inbox)}")

## 2. Broadcast Communication

One agent sends a message to all other agents in the system.

In [ ]:
class MessageBus:
    """Central message bus for broadcast communication."""
    
    def __init__(self):
        self.agents: Dict[str, CommunicatingAgent] = {}
        self.message_log: List[Message] = []
        
    def register(self, agent: CommunicatingAgent):
        """Register an agent with the message bus."""
        self.agents[agent.name] = agent
        print(f"✓ Registered {agent.name}")
        
    def broadcast(self, sender: str, content: str, exclude_sender: bool = True):
        """Broadcast a message to all agents."""
        print(f"\n📢 BROADCAST from [{sender}]: {content}")
        
        for agent_name, agent in self.agents.items():
            if exclude_sender and agent_name == sender:
                continue
                
            msg = Message(
                sender=sender,
                recipient=agent_name,
                content=content,
                message_type="broadcast"
            )
            agent.receive(msg)
            self.message_log.append(msg)
            
    def send(self, sender: str, recipient: str, content: str):
        """Send a point-to-point message via the bus."""
        if recipient in self.agents:
            msg = Message(sender=sender, recipient=recipient, content=content)
            self.agents[recipient].receive(msg)
            self.message_log.append(msg)
        else:
            print(f"❌ Error: {recipient} not found")


# Example: Broadcast pattern
bus = MessageBus()

# Create and register agents
coordinator = CommunicatingAgent("Coordinator")
worker1 = CommunicatingAgent("Worker1")
worker2 = CommunicatingAgent("Worker2")
worker3 = CommunicatingAgent("Worker3")

for agent in [coordinator, worker1, worker2, worker3]:
    bus.register(agent)

# Coordinator broadcasts a task
bus.broadcast("Coordinator", "New task available: Process dataset batch #42")

print(f"\n📊 Message log: {len(bus.message_log)} messages")

## 3. Publish-Subscribe Pattern

Agents subscribe to specific topics and receive relevant messages.

In [ ]:
class PubSubBus:
    """Publish-Subscribe message bus."""
    
    def __init__(self):
        self.subscriptions: Dict[str, List[CommunicatingAgent]] = {}
        self.agents: Dict[str, CommunicatingAgent] = {}
        
    def register(self, agent: CommunicatingAgent):
        """Register an agent."""
        self.agents[agent.name] = agent
        
    def subscribe(self, agent_name: str, topic: str):
        """Subscribe an agent to a topic."""
        if topic not in self.subscriptions:
            self.subscriptions[topic] = []
        
        agent = self.agents.get(agent_name)
        if agent and agent not in self.subscriptions[topic]:
            self.subscriptions[topic].append(agent)
            print(f"✓ [{agent_name}] subscribed to '{topic}'")
            
    def publish(self, sender: str, topic: str, content: str):
        """Publish a message to a topic."""
        print(f"\n📰 PUBLISH to '{topic}' from [{sender}]: {content}")
        
        if topic in self.subscriptions:
            for agent in self.subscriptions[topic]:
                msg = Message(
                    sender=sender,
                    recipient=agent.name,
                    content=content,
                    message_type=topic
                )
                agent.receive(msg)
        else:
            print(f"  (No subscribers for '{topic}')")


# Example: Pub-Sub pattern
pubsub = PubSubBus()

# Create specialized agents
data_collector = CommunicatingAgent("DataCollector")
data_processor = CommunicatingAgent("DataProcessor")
analytics = CommunicatingAgent("Analytics")
logger = CommunicatingAgent("Logger")

# Register agents
for agent in [data_collector, data_processor, analytics, logger]:
    pubsub.register(agent)

# Subscribe to topics
pubsub.subscribe("DataProcessor", "data.raw")
pubsub.subscribe("Analytics", "data.processed")
pubsub.subscribe("Logger", "data.raw")
pubsub.subscribe("Logger", "data.processed")
pubsub.subscribe("Logger", "errors")

# Publish messages
pubsub.publish("DataCollector", "data.raw", "New data batch collected: 1000 records")
pubsub.publish("DataProcessor", "data.processed", "Processed 1000 records, 15 anomalies found")
pubsub.publish("DataProcessor", "errors", "Warning: 3 records had missing fields")

print(f"\n📊 Subscription summary:")
for topic, subscribers in pubsub.subscriptions.items():
    print(f"  '{topic}': {[s.name for s in subscribers]}")

## 4. Request-Reply Pattern

Synchronous communication where one agent requests information and waits for a reply.

In [ ]:
class RequestReplyAgent(CommunicatingAgent):
    """Agent that can handle request-reply interactions."""
    
    def __init__(self, name: str):
        super().__init__(name)
        self.request_handlers: Dict[str, Callable] = {}
        
    def register_handler(self, request_type: str, handler: Callable):
        """Register a handler for a specific request type."""
        self.request_handlers[request_type] = handler
        
    def handle_request(self, message: Message) -> Message:
        """Process a request and generate a reply."""
        request_type = message.metadata.get('request_type', 'default')
        
        if request_type in self.request_handlers:
            result = self.request_handlers[request_type](message.content)
        else:
            result = f"Unknown request type: {request_type}"
            
        return Message(
            sender=self.name,
            recipient=message.sender,
            content=result,
            message_type="reply",
            metadata={'in_reply_to': message.timestamp}
        )


# Example: Request-Reply
database_agent = RequestReplyAgent("DatabaseAgent")
client_agent = RequestReplyAgent("ClientAgent")

# Register handlers
def query_handler(query: str) -> str:
    return f"Query results for '{query}': [Record1, Record2, Record3]"

def count_handler(query: str) -> str:
    return f"Count for '{query}': 42 records"

database_agent.register_handler('query', query_handler)
database_agent.register_handler('count', count_handler)

# Client sends request
request = Message(
    sender="ClientAgent",
    recipient="DatabaseAgent",
    content="SELECT * FROM users WHERE active=true",
    message_type="request",
    metadata={'request_type': 'query'}
)

print("📤 Request:")
database_agent.receive(request)

# Database processes and replies
reply = database_agent.handle_request(request)

print("\n📥 Reply:")
client_agent.receive(reply)

print(f"\n✓ Round-trip communication complete")

## 5. Blackboard Pattern

Shared knowledge repository where agents read and write information.

In [ ]:
class Blackboard:
    """Shared knowledge base for multi-agent collaboration."""
    
    def __init__(self):
        self.data: Dict[str, Any] = {}
        self.history: List[Dict] = []
        
    def write(self, key: str, value: Any, author: str):
        """Write data to the blackboard."""
        self.data[key] = value
        self.history.append({
            'action': 'write',
            'key': key,
            'author': author,
            'timestamp': datetime.now()
        })
        print(f"✍️  [{author}] wrote '{key}': {str(value)[:50]}")
        
    def read(self, key: str, reader: str) -> Optional[Any]:
        """Read data from the blackboard."""
        value = self.data.get(key)
        self.history.append({
            'action': 'read',
            'key': key,
            'reader': reader,
            'timestamp': datetime.now()
        })
        print(f"👁️  [{reader}] read '{key}'")
        return value
        
    def get_all_keys(self) -> List[str]:
        """Get all available keys."""
        return list(self.data.keys())


# Example: Collaborative problem solving
blackboard = Blackboard()

# Multiple agents contribute to solving a problem
blackboard.write('problem', 'Optimize neural network architecture', 'UserAgent')
blackboard.write('constraints', {'max_params': '10M', 'latency': '<100ms'}, 'UserAgent')

# Research agent adds findings
blackboard.write('sota_architectures', ['EfficientNet', 'MobileNet', 'SqueezeNet'], 'ResearchAgent')

# Analysis agent processes
architectures = blackboard.read('sota_architectures', 'AnalysisAgent')
constraints = blackboard.read('constraints', 'AnalysisAgent')
blackboard.write('analysis_results', {'recommended': 'MobileNetV3', 'confidence': 0.87}, 'AnalysisAgent')

# Design agent creates solution
analysis = blackboard.read('analysis_results', 'DesignAgent')
blackboard.write('final_design', {'architecture': 'MobileNetV3', 'optimizations': ['pruning', 'quantization']}, 'DesignAgent')

print(f"\n📊 Blackboard state:")
print(f"  Keys: {blackboard.get_all_keys()}")
print(f"  History: {len(blackboard.history)} actions")

## 6. Visualizing Communication Patterns

Let's visualize the message flow between agents:

In [ ]:
# Create a communication scenario
messages = [
    {'from': 'User', 'to': 'Orchestrator', 'content': 'Build a web app'},
    {'from': 'Orchestrator', 'to': 'PlannerAgent', 'content': 'Create project plan'},
    {'from': 'PlannerAgent', 'to': 'Orchestrator', 'content': 'Plan ready'},
    {'from': 'Orchestrator', 'to': 'DesignAgent', 'content': 'Design UI'},
    {'from': 'Orchestrator', 'to': 'BackendAgent', 'content': 'Build API'},
    {'from': 'DesignAgent', 'to': 'Orchestrator', 'content': 'UI design complete'},
    {'from': 'BackendAgent', 'to': 'Orchestrator', 'content': 'API ready'},
    {'from': 'Orchestrator', 'to': 'TestAgent', 'content': 'Run tests'},
    {'from': 'TestAgent', 'to': 'Orchestrator', 'content': 'All tests passed'},
    {'from': 'Orchestrator', 'to': 'User', 'content': 'Project complete'},
]

visualize_communication_flow(messages, title="Multi-Agent Web Development Flow")

In [ ]:
# Create interaction heatmap
interaction_matrix = {
    'Orchestrator': {'PlannerAgent': 2, 'DesignAgent': 2, 'BackendAgent': 2, 'TestAgent': 2},
    'PlannerAgent': {'Orchestrator': 1},
    'DesignAgent': {'Orchestrator': 1},
    'BackendAgent': {'Orchestrator': 1},
    'TestAgent': {'Orchestrator': 1},
}

plot_agent_interactions(interaction_matrix, title="Agent Communication Heatmap")

## Pattern Comparison

| Pattern | Use Case | Pros | Cons |
|---------|----------|------|------|
| **Point-to-Point** | Direct agent collaboration | Simple, predictable | Tight coupling |
| **Broadcast** | System-wide announcements | All agents informed | Message overhead |
| **Pub-Sub** | Event-driven systems | Loose coupling, scalable | Complex routing |
| **Request-Reply** | Synchronous queries | Guaranteed response | Blocking |
| **Blackboard** | Collaborative problem-solving | Flexible, asynchronous | Potential conflicts |
| **Contract Net** | Task allocation | Optimal resource use | Negotiation overhead |

## Exercise: Design a Communication Protocol

Design a communication protocol for a multi-agent system that:
1. Has a coordinator and 3 worker agents
2. Workers can request tasks from the coordinator
3. Workers report progress back
4. Coordinator can broadcast priority updates

Which patterns would you combine?

In [ ]:
# Your communication protocol here

# Hint: You might want to combine Request-Reply (for task requests)
# with Broadcast (for priority updates) and Pub-Sub (for progress reports)

## Summary

In this notebook, we explored:

- ✓ Different communication patterns for multi-agent systems
- ✓ Implementation of message buses and blackboards
- ✓ Trade-offs between different patterns
- ✓ Visualization of agent communication

**Next:** In Notebook 3, we'll dive into orchestration strategies - how to coordinate multiple agents to accomplish complex tasks.